# Aula 04 — Diagnósticos estatísticos e aderência à NBR 14653

Nesta aula, vamos ler a saída da Aula 3 e avaliar os diagnósticos do modelo.

## Objetivos

- consumir a saída da Aula 3;
- gerar o relatório de diagnósticos;
- analisar significância e resíduos;
- concluir a sequência do treinamento.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import resolve_project_root
from servicos.regressao import fit_ols_regression
from servicos.diagnosticos import (
    build_nbr_diagnostics_report,
    format_coefficients_for_display,
    format_diagnostics_for_display,
)

In [2]:
TARGET_COLUMN: Final[str] = 'preco'
PREFERRED_FEATURES: Final[tuple[str, ...]] = (
    'areaprivativa',
    'vagas',
    'distanciacentrokm',
    'dist_praia',
)


def locate_input_file(project_root: Path) -> Path:
    candidate = project_root / 'data' / 'output' / 'aula_03_amostra_com_residuos.csv'
    if candidate.exists():
        return candidate
    raise FileNotFoundError('Saída da Aula 3 não encontrada em data/output/.')

## Etapa 1 — Ler a saída da Aula 3

In [3]:
project_root = resolve_project_root()
input_path = locate_input_file(project_root)
df_prepared = pd.read_csv(input_path)
print('Base carregada da Aula 3:', df_prepared.shape)
df_prepared.head()

Base carregada da Aula 3: (17, 10)


,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink,valor_unitario,valor_ajustado,residuo
0,AP-001,750000.0,85.0,2,5.0,1.2,https://portalimoveis.com.br/anuncio/001,8823.529412,758141.189993,-8141.189993
1,AP-002,820000.0,92.5,2,8.0,1.8,https://portalimoveis.com.br/anuncio/002,8864.864865,831270.941113,-11270.941113
2,AP-003,690000.0,78.0,1,12.0,2.5,https://portalimoveis.com.br/anuncio/003,8846.153846,696955.639390,-6955.639390
3,AP-005,580000.0,65.0,1,20.0,4.2,https://portalimoveis.com.br/anuncio/005,8923.076923,576882.712251,3117.287749
4,AP-006,950000.0,105.0,2,4.0,1.5,https://portalimoveis.com.br/anuncio/006,9047.619048,949981.998122,18.001878


## Etapa 2 — Ajustar e diagnosticar o modelo

In [4]:
available_features = [column for column in PREFERRED_FEATURES if column in df_prepared.columns]
artifacts = fit_ols_regression(
    df=df_prepared,
    target_col=TARGET_COLUMN,
    feature_columns=available_features,
    add_intercept=True,
)

diagnostics_report, coefficients_report, summary = build_nbr_diagnostics_report(
    artifacts,
    minimum_adjusted_r_squared=0.70,
    alpha_f=0.05,
    alpha_t=0.10,
    alpha_shapiro=0.05,
    dw_lower_bound=1.5,
    dw_upper_bound=2.5,
)

## Etapa 3 — Ler os diagnósticos

In [5]:
format_diagnostics_for_display(diagnostics_report)

,pressuposto,metrica_teste,valor_obtido,criterio_aceitacao,status
0,Poder Explicativo,R2 Ajustado,0.996532,R2 Ajustado >= 0.70,APROVADO
1,Significancia Global,Teste F p-valor,0.000000,p-valor F < 0.05,APROVADO
2,Normalidade dos Residuos,Shapiro-Wilk p-valor,0.965756,p-valor W > 0.05,NORMAL
3,Independencia de Residuos,Durbin-Watson,1.026030,Entre 1.5 e 2.5,ALERTA


In [6]:
format_coefficients_for_display(coefficients_report)

,variavel_explicativa,coeficiente,erro_padrao,estatistica_t,p_valor_t,significativo_10pct
0,const,-51937.888738,33119.885658,-1.568178,0.116840,NAO
1,areaprivativa,9555.442004,372.185782,25.673850,0.000000,SIM
2,vagas,-2530.681903,6702.507753,-0.377572,0.705748,NAO
3,distanciacentrokm,2439.893480,3901.049563,0.625445,0.531679,NAO


## Etapa 4 — Resumo final

In [7]:
summary

{'status_geral_modelo': 'APROVADO',
 'itens_aprovados_ou_normais': 3,
 'itens_avaliados': 4,
 'proporcao_aprovacao': 0.75,
 'teste_normalidade_observacao': 'Teste executado com scipy.stats.shapiro.',
 'durbin_watson_valor': 1.0260301322158238,
 'durbin_watson_faixa': '1.5 a 2.5',
 'coeficientes_significativos_10pct': 1,
 'coeficientes_totais': 4}

In [8]:
print(f"Status geral do modelo: {summary['status_geral_modelo']}")
print(
    'Itens aprovados ou normais: '
    f"{summary['itens_aprovados_ou_normais']} de {summary['itens_avaliados']}"
)
print(f"Proporção de aprovação: {summary['proporcao_aprovacao']:.2%}")
print(
    'Coeficientes significativos a 10%: '
    f"{summary['coeficientes_significativos_10pct']} de {summary['coeficientes_totais']}"
)
print('Observação sobre normalidade:', summary['teste_normalidade_observacao'])
print('Faixa usada para Durbin-Watson:', summary['durbin_watson_faixa'])
print(f"Valor observado de Durbin-Watson: {summary['durbin_watson_valor']:.6f}")

Status geral do modelo: APROVADO
Itens aprovados ou normais: 3 de 4
Proporção de aprovação: 75.00%
Coeficientes significativos a 10%: 1 de 4
Observação sobre normalidade: Teste executado com scipy.stats.shapiro.
Faixa usada para Durbin-Watson: 1.5 a 2.5
Valor observado de Durbin-Watson: 1.026030


## Conclusão

A Aula 4 fecha o ciclo do treinamento e consolida a leitura técnica do modelo.